In [35]:
from pathlib import Path
import json

LABEL_DIR = Path("../data/aihub/labels")
IMAGE_DIR = Path("../data/aihub/images")

# JSON 하나 찾기
sample_json = next(LABEL_DIR.rglob("*.json"))

print("JSON:")
print(sample_json)

# JSON 읽기
with open(sample_json, "r", encoding="utf-8") as f:
    data = json.load(f)

# JSON의 image 정보 확인
print("\nJSON image 정보:")
for img in data.get("images", [])[:5]:
    print(img)

JSON:
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000250\K-000250-000573-002483-006192_0_2_0_2_70_000_200.json

JSON image 정보:
{'file_name': 'K-000250-000573-002483-006192_0_2_0_2_70_000_200.png', 'width': 976, 'height': 1280, 'imgfile': 'K-000250-000573-002483-006192_0_2_0_2_70_000_200.png', 'drug_N': 'K-000250', 'drug_S': '정상알약', 'back_color': '연회색 배경', 'drug_dir': '앞면', 'light_color': '주백색', 'camera_la': 70, 'camera_lo': 0, 'size': 200, 'dl_idx': '249', 'dl_mapping_code': 'K-000250', 'dl_name': '마그밀정(수산화마그네슘)', 'dl_name_en': 'Magmil Tab. 500mg', 'img_key': 'http://connectdi.com/design/img/drug/1NgBQQRiuc_.jpg', 'dl_material': '수산화마그네슘', 'dl_material_en': 'Magnesium Hydroxide', 'dl_custom_shape': '정제, 저작정', 'dl_company': '삼남제약(주)', 'dl_company_en': 'Samnam', 'di_company_mf': '', 'di_company_mf_en': '', 'item_seq': 197400246, 'di_item_permit_date': '19740902', 'di_class_no': '[02340]제산제', 'di_etc_otc_code': '일반의약품', 'di_edi_code': '653700240,A11800371', 'chart': '흰색의 원형정

In [36]:
# JSON이 들어있는 _json 폴더명
json_folder = next(
    part for part in sample_json.parts
    if part.endswith("_json")
)

# _json 제거 → 이미지 폴더에서 찾을 K 폴더명
k_folder_name = json_folder.replace("_json", "")

print("\n찾을 이미지 폴더:")
print(k_folder_name)

# images 아래에서 해당 폴더 검색
image_dirs = list(IMAGE_DIR.rglob(k_folder_name))

print("\n발견된 이미지 폴더:")
for p in image_dirs[:10]:
    print(p)


찾을 이미지 폴더:
K-000250-000573-002483-006192

발견된 이미지 폴더:
..\data\aihub\images\TS_1_조합\K-000250-000573-002483-006192


1. 전체 JSON ↔ 이미지 매칭 검사

In [37]:
from pathlib import Path
import json

LABEL_DIR = Path("../data/aihub/labels")
IMAGE_DIR = Path("../data/aihub/images")

# 전체 JSON 탐색
json_files = sorted(LABEL_DIR.rglob("*.json"))

print("전체 JSON:", len(json_files))

matched = 0
missing = []
parse_errors = []

for json_path in json_files:

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

    except Exception as e:
        parse_errors.append({
            "json_path": str(json_path),
            "error": str(e)
        })
        continue

    # JSON 파일이 들어있는 _json 폴더
    json_folder = next(
        (
            part
            for part in json_path.parts
            if part.endswith("_json")
        ),
        None
    )

    if json_folder is None:
        continue

    # _json 제거 → 이미지 폴더명
    image_folder_name = json_folder[:-5]

    # 이미지 폴더
    image_dirs = list(
        IMAGE_DIR.rglob(image_folder_name)
    )

    if not image_dirs:
        missing.append({
            "json_path": str(json_path),
            "reason": "이미지 폴더 없음",
            "image_folder": image_folder_name
        })
        continue

    image_dir = image_dirs[0]

    # JSON 내부의 이미지 파일 확인
    for img in data.get("images", []):

        file_name = img.get("file_name")

        if not file_name:
            missing.append({
                "json_path": str(json_path),
                "reason": "file_name 없음"
            })
            continue

        image_path = image_dir / file_name

        if image_path.exists():
            matched += 1
        else:
            missing.append({
                "json_path": str(json_path),
                "reason": "실제 이미지 파일 없음",
                "image_path": str(image_path)
            })

print("=" * 60)
print("JSON ↔ 이미지 매칭 결과")
print("=" * 60)
print(f"전체 JSON          : {len(json_files):,}")
print(f"매칭된 이미지      : {matched:,}")
print(f"매칭 실패          : {len(missing):,}")
print(f"JSON 파싱 오류     : {len(parse_errors):,}")

전체 JSON: 40499


KeyboardInterrupt: 

In [38]:
from pathlib import Path
import json

LABEL_DIR = Path("../data/aihub/labels")
IMAGE_DIR = Path("../data/aihub/images")

# 이미지 폴더를 딱 한 번만 수집
image_dirs = {
    p.name: p
    for p in IMAGE_DIR.rglob("*")
    if p.is_dir()
}

print("이미지 폴더 수:", len(image_dirs))

이미지 폴더 수: 3510


In [39]:
json_files = list(LABEL_DIR.rglob("*.json"))

matched = 0
missing_folder = []
missing_file = []
parse_errors = []

for json_path in json_files:

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        parse_errors.append((json_path, str(e)))
        continue

    # K-...._json
    json_folder = next(
        part for part in json_path.parts
        if part.endswith("_json")
    )

    # K-.... 
    image_folder_name = json_folder[:-5]

    image_dir = image_dirs.get(image_folder_name)

    if image_dir is None:
        missing_folder.append({
            "json": str(json_path),
            "folder": image_folder_name
        })
        continue

    for img in data.get("images", []):

        file_name = img.get("file_name")

        if not file_name:
            continue

        if (image_dir / file_name).exists():
            matched += 1
        else:
            missing_file.append({
                "json": str(json_path),
                "image": str(image_dir / file_name)
            })

print("=" * 50)
print("매칭 결과")
print("=" * 50)
print("전체 JSON:", len(json_files))
print("매칭 이미지:", matched)
print("이미지 폴더 없음:", len(missing_folder))
print("이미지 파일 없음:", len(missing_file))
print("JSON 파싱 오류:", len(parse_errors))

KeyboardInterrupt: 

2. 매칭 실패한 것 확인

In [ ]:
import pandas as pd

missing_df = pd.DataFrame(missing)

print("매칭 실패:", len(missing_df))

missing_df.head(20)

In [40]:
from pathlib import Path
import json

LABEL_DIR = Path("../data/aihub/labels")
IMAGE_DIR = Path("../data/aihub/images")

# 이미지 파일을 한 번만 인덱싱
print("이미지 파일 인덱싱 중...")

image_files = {}

for p in IMAGE_DIR.rglob("*.png"):
    image_files[(p.parent.name, p.name)] = p

print("이미지 파일 수:", len(image_files))

이미지 파일 인덱싱 중...
이미지 파일 수: 14012


In [41]:
json_files = list(LABEL_DIR.rglob("*.json"))

matched = 0
missing_folder = []
missing_file = []
parse_errors = []

for i, json_path in enumerate(json_files, 1):

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

    except Exception as e:
        parse_errors.append((str(json_path), str(e)))
        continue

    # ..._json 폴더 찾기
    json_folder = next(
        part for part in json_path.parts
        if part.endswith("_json")
    )

    image_folder_name = json_folder[:-5]

    for img in data.get("images", []):

        file_name = img.get("file_name")

        if not file_name:
            continue

        key = (image_folder_name, file_name)

        if key in image_files:
            matched += 1
        else:
            missing_file.append({
                "json": str(json_path),
                "folder": image_folder_name,
                "image": file_name
            })

    # 진행 상황
    if i % 1000 == 0:
        print(f"{i:,} / {len(json_files):,} JSON 검사 완료")

print("\n" + "=" * 50)
print("매칭 결과")
print("=" * 50)
print("전체 JSON:", len(json_files))
print("매칭 이미지:", matched)
print("이미지 파일 없음:", len(missing_file))
print("JSON 파싱 오류:", len(parse_errors))

1,000 / 40,499 JSON 검사 완료
2,000 / 40,499 JSON 검사 완료
3,000 / 40,499 JSON 검사 완료
4,000 / 40,499 JSON 검사 완료
5,000 / 40,499 JSON 검사 완료
6,000 / 40,499 JSON 검사 완료
7,000 / 40,499 JSON 검사 완료
8,000 / 40,499 JSON 검사 완료
9,000 / 40,499 JSON 검사 완료
10,000 / 40,499 JSON 검사 완료
11,000 / 40,499 JSON 검사 완료
12,000 / 40,499 JSON 검사 완료
13,000 / 40,499 JSON 검사 완료
14,000 / 40,499 JSON 검사 완료
15,000 / 40,499 JSON 검사 완료
16,000 / 40,499 JSON 검사 완료
17,000 / 40,499 JSON 검사 완료
18,000 / 40,499 JSON 검사 완료
19,000 / 40,499 JSON 검사 완료
20,000 / 40,499 JSON 검사 완료
21,000 / 40,499 JSON 검사 완료
22,000 / 40,499 JSON 검사 완료
23,000 / 40,499 JSON 검사 완료
24,000 / 40,499 JSON 검사 완료
25,000 / 40,499 JSON 검사 완료
26,000 / 40,499 JSON 검사 완료
27,000 / 40,499 JSON 검사 완료
28,000 / 40,499 JSON 검사 완료
29,000 / 40,499 JSON 검사 완료
30,000 / 40,499 JSON 검사 완료
31,000 / 40,499 JSON 검사 완료
32,000 / 40,499 JSON 검사 완료
33,000 / 40,499 JSON 검사 완료
34,000 / 40,499 JSON 검사 완료
35,000 / 40,499 JSON 검사 완료
36,000 / 40,499 JSON 검사 완료
37,000 / 40,499 JSON 검사 완료
38,000 / 4

In [42]:
print("\n[이미지 파일 없음]")
for x in missing_file[:50]:
    print(x)

print("\n[JSON 파싱 오류]")
for x in parse_errors:
    print(x)


[이미지 파일 없음]
{'json': '..\\data\\aihub\\labels\\K-003544-012247-016548-029345_json\\K-003544\\K-003544-012247-016548-029345_0_2_0_2_70_000_200.json', 'folder': 'K-003544-012247-016548-029345', 'image': 'K-003544-012247-016548-029345_0_2_0_2_70_000_200.png'}
{'json': '..\\data\\aihub\\labels\\K-003544-012247-016548-029345_json\\K-003544\\K-003544-012247-016548-029345_0_2_0_2_75_000_200.json', 'folder': 'K-003544-012247-016548-029345', 'image': 'K-003544-012247-016548-029345_0_2_0_2_75_000_200.png'}
{'json': '..\\data\\aihub\\labels\\K-003544-012247-016548-029345_json\\K-003544\\K-003544-012247-016548-029345_0_2_0_2_90_000_200.json', 'folder': 'K-003544-012247-016548-029345', 'image': 'K-003544-012247-016548-029345_0_2_0_2_90_000_200.png'}
{'json': '..\\data\\aihub\\labels\\K-003544-012247-016548-029345_json\\K-012247\\K-003544-012247-016548-029345_0_2_0_2_70_000_200.json', 'folder': 'K-003544-012247-016548-029345', 'image': 'K-003544-012247-016548-029345_0_2_0_2_70_000_200.png'}
{'json'

In [43]:
print("=== 이미지 파일 없음 36개 ===")

for x in missing_file:
    print(f"\nJSON:   {x['json']}")
    print(f"폴더:   {x['folder']}")
    print(f"파일:   {x['image']}")

    # 해당 폴더 자체가 존재하는지 확인
    folder_path = IMAGE_DIR / x["folder"]

    print("폴더 존재:", folder_path.exists())

    if folder_path.exists():
        # 해당 폴더에 png가 몇 개 있는지
        pngs = list(folder_path.glob("*.png"))
        print("PNG 개수:", len(pngs))

        # 비슷한 파일명 찾아보기
        target = Path(x["image"]).stem
        similar = [p.name for p in pngs if target[:20] in p.name]

        print("비슷한 파일:", similar[:5])

=== 이미지 파일 없음 36개 ===

JSON:   ..\data\aihub\labels\K-003544-012247-016548-029345_json\K-003544\K-003544-012247-016548-029345_0_2_0_2_70_000_200.json
폴더:   K-003544-012247-016548-029345
파일:   K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
폴더 존재: False

JSON:   ..\data\aihub\labels\K-003544-012247-016548-029345_json\K-003544\K-003544-012247-016548-029345_0_2_0_2_75_000_200.json
폴더:   K-003544-012247-016548-029345
파일:   K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
폴더 존재: False

JSON:   ..\data\aihub\labels\K-003544-012247-016548-029345_json\K-003544\K-003544-012247-016548-029345_0_2_0_2_90_000_200.json
폴더:   K-003544-012247-016548-029345
파일:   K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
폴더 존재: False

JSON:   ..\data\aihub\labels\K-003544-012247-016548-029345_json\K-012247\K-003544-012247-016548-029345_0_2_0_2_70_000_200.json
폴더:   K-003544-012247-016548-029345
파일:   K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
폴더 존재: False

JSON:   ..\data\aihub\labels\K-00

In [45]:
print("=== 누락 이미지 실제 존재 여부 검사 ===")

for x in missing_file:

    file_name = x["image"]

    # images 전체에서 동일한 파일명을 검색
    found = list(IMAGE_DIR.rglob(file_name))

    print("\n파일:", file_name)

    if found:
        print("✅ 실제 이미지 존재")
        print("   위치:", found[0])
    else:
        print("❌ 이미지 자체가 없음")

=== 누락 이미지 실제 존재 여부 검사 ===

파일: K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029451_0_2_0_2_70_000_200.png
❌ 이미지 자체가 없음

파일: K-003544-012247-016548-029451_0_2_0_2_75_000_

In [44]:
print("=== JSON 파싱 오류 ===")

for path, error in parse_errors:
    print("\n파일:", path)
    print("오류:", error)

=== JSON 파싱 오류 ===

파일: ..\data\aihub\labels\K-005094-005886-012778-037777_json\K-037777\._K-005094-005886-012778-037777_0_2_0_2_90_000_200.json
오류: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte

파일: ..\data\aihub\labels\K-012247-016551-033009-044199_json\K-016551-오류\K-012247-016551-033009-044199_0_2_0_2_90_000_200.json
오류: Expecting value: line 59 column 12 (char 1571)


In [4]:
from pathlib import Path
from collections import Counter
import json

# 경로
aihub_root = Path("../data/aihub")
json_root = aihub_root / "labels"
image_root = aihub_root / "images"

print("=" * 60)
print("AI-Hub 데이터 오류 통계")
print("=" * 60)

# =========================================================
# 1. 전체 JSON 수집
# =========================================================

json_files = list(json_root.rglob("*.json"))

print(f"\n전체 JSON 파일 수 : {len(json_files):,}")


# =========================================================
# 2. JSON 파싱 검사
# =========================================================

valid_json = []
parse_errors = []

for path in json_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        valid_json.append((path, data))

    except Exception as e:
        parse_errors.append((path, type(e).__name__, str(e)))


print(f"정상 JSON          : {len(valid_json):,}")
print(f"JSON 파싱 오류     : {len(parse_errors):,}")


# =========================================================
# 3. JSON 오류 종류
# =========================================================

error_types = Counter(
    error_type for _, error_type, _ in parse_errors
)

print("\n[JSON 오류 종류]")

for error_type, count in error_types.items():
    print(f"{error_type:<25} {count:,}")


# =========================================================
# 4. JSON → 이미지 존재 여부
# =========================================================

missing_images = []
existing_images = []

# 이미지 파일명을 빠르게 찾기 위해
# 이미지 전체 목록을 미리 만들어둠
image_map = {}

for image_path in image_root.rglob("*.png"):
    image_map.setdefault(image_path.name, []).append(image_path)


for json_path, data in valid_json:

    image_name = json_path.stem + ".png"

    if image_name in image_map:

        existing_images.append({
            "json": str(json_path),
            "image": image_name
        })

    else:

        missing_images.append({
            "json": str(json_path),
            "image": image_name
        })


print("\n" + "=" * 60)
print("[이미지 존재 여부]")
print("=" * 60)

print(f"이미지 존재 JSON  : {len(existing_images):,}")
print(f"이미지 누락 JSON  : {len(missing_images):,}")


# =========================================================
# 5. 실제 누락 이미지 종류
# =========================================================

missing_image_names = sorted(
    set(item["image"] for item in missing_images)
)

print("\n" + "=" * 60)
print("[중복 제거한 실제 누락 이미지]")
print("=" * 60)

print(f"실제 누락 이미지 종류 : {len(missing_image_names):,}")

for image_name in missing_image_names:
    print(image_name)


# =========================================================
# 6. 최종 요약
# =========================================================

print("\n" + "=" * 60)
print("[최종 요약]")
print("=" * 60)

print(f"전체 JSON              : {len(json_files):,}")
print(f"정상 JSON              : {len(valid_json):,}")
print(f"JSON 파싱 오류         : {len(parse_errors):,}")
print(f"이미지 존재 JSON       : {len(existing_images):,}")
print(f"이미지 누락 JSON       : {len(missing_images):,}")
print(f"실제 누락 이미지 종류  : {len(missing_image_names):,}")
print(f"전체 이미지 파일       : {len(image_map):,}")

AI-Hub 데이터 오류 통계

전체 JSON 파일 수 : 40,499


: 

In [2]:
from pathlib import Path
import json

json_root = Path("../data/aihub/labels")

# JSON 하나만 찾기
sample_path = next(json_root.rglob("*.json"))

print("파일:")
print(sample_path)

try:
    with open(sample_path, "r", encoding="utf-8") as f:
        sample_data = json.load(f)

    print("\n=== JSON 구조 ===")
    print(json.dumps(sample_data, ensure_ascii=False, indent=2)[:5000])

except Exception as e:
    print("이 JSON은 오류가 있습니다.")
    print(type(e).__name__, ":", e)

파일:
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000250\K-000250-000573-002483-006192_0_2_0_2_70_000_200.json

=== JSON 구조 ===
{
  "images": [
    {
      "file_name": "K-000250-000573-002483-006192_0_2_0_2_70_000_200.png",
      "width": 976,
      "height": 1280,
      "imgfile": "K-000250-000573-002483-006192_0_2_0_2_70_000_200.png",
      "drug_N": "K-000250",
      "drug_S": "정상알약",
      "back_color": "연회색 배경",
      "drug_dir": "앞면",
      "light_color": "주백색",
      "camera_la": 70,
      "camera_lo": 0,
      "size": 200,
      "dl_idx": "249",
      "dl_mapping_code": "K-000250",
      "dl_name": "마그밀정(수산화마그네슘)",
      "dl_name_en": "Magmil Tab. 500mg",
      "img_key": "http://connectdi.com/design/img/drug/1NgBQQRiuc_.jpg",
      "dl_material": "수산화마그네슘",
      "dl_material_en": "Magnesium Hydroxide",
      "dl_custom_shape": "정제, 저작정",
      "dl_company": "삼남제약(주)",
      "dl_company_en": "Samnam",
      "di_company_mf": "",
      "di_company_mf_en": "",
      

In [5]:
from pathlib import Path

# 프로젝트의 data 폴더 아래 JSON 찾기
project_root = Path("../data")

json_files = list(project_root.rglob("*.json"))

print(f"JSON 파일 수: {len(json_files):,}")

for path in json_files[:20]:
    print(path)

JSON 파일 수: 41,262
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000250\K-000250-000573-002483-006192_0_2_0_2_70_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000250\K-000250-000573-002483-006192_0_2_0_2_75_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000250\K-000250-000573-002483-006192_0_2_0_2_90_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000573\K-000250-000573-002483-006192_0_2_0_2_70_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000573\K-000250-000573-002483-006192_0_2_0_2_75_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-000573\K-000250-000573-002483-006192_0_2_0_2_90_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-002483\K-000250-000573-002483-006192_0_2_0_2_70_000_200.json
..\data\aihub\labels\K-000250-000573-002483-006192_json\K-002483\K-000250-000573-002483-006192_0_2_0_2_75_000_200.json
..\data\aihub\labels\K-000250-

In [7]:
from pathlib import Path

data_root = Path("../data")

for p in data_root.iterdir():
    print(p.name, "/" if p.is_dir() else "")

aihub /
processed /
raw /
README.md 


In [4]:
import json

# 기존 프로젝트 JSON 파일 경로로 수정
json_path = "여기에_기존_JSON_경로"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data["categories"], ensure_ascii=False, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '여기에_기존_JSON_경로'

In [8]:
from pathlib import Path

raw_root = Path("../data/raw")

print("raw 폴더 구조")
print("=" * 50)

for p in raw_root.rglob("*"):
    if p.is_file() and p.suffix.lower() == ".json":
        print(p)

raw 폴더 구조
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-001900\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-001900\K-001900-016548-019607-029451_0_2_0_2_75_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-001900\K-001900-016548-019607-029451_0_2_0_2_90_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-016548\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-016548\K-001900-016548-019607-029451_0_2_0_2_75_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-016548\K-001900-016548-019607-029451_0_2_0_2_90_000_200.json
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-0

In [9]:
import json
from pathlib import Path

json_path = next(
    Path("../data/raw/sprint_ai_project1_data/train_annotations")
    .rglob("*.json")
)

print("확인할 파일:")
print(json_path)

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\n최상위 키:")
print(data.keys())

print("\ncategories:")
for category in data.get("categories", []):
    print(category)

확인할 파일:
..\data\raw\sprint_ai_project1_data\train_annotations\K-001900-016548-019607-029451_json\K-001900\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json

최상위 키:
dict_keys(['images', 'type', 'annotations', 'categories'])

categories:
{'supercategory': 'pill', 'id': 1900, 'name': '보령부스파정 5mg'}


In [10]:
import json
from pathlib import Path

ann_root = Path("../data/raw/sprint_ai_project1_data/train_annotations")

categories = {}

for json_path in ann_root.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for cat in data.get("categories", []):
            categories[cat["id"]] = cat["name"]

    except Exception:
        continue

print(f"총 클래스 수: {len(categories)}")

for category_id, name in sorted(categories.items()):
    print(category_id, ":", name)

총 클래스 수: 56
1900 : 보령부스파정 5mg
2483 : 뮤테란캡슐 100mg
3351 : 일양하이트린정 2mg
3483 : 기넥신에프정(은행엽엑스)(수출용)
3544 : 무코스타정(레바미피드)(비매품)
3743 : 알드린정
3832 : 뉴로메드정(옥시라세탐)
4543 : 에어탈정(아세클로페낙)
12081 : 리렉스펜정 300mg/PTP
12247 : 아빌리파이정 10mg
12778 : 다보타민큐정 10mg/병
13395 : 써스펜8시간이알서방정 650mg
13900 : 에빅사정(메만틴염산염)(비매품)
16232 : 리피토정 20mg
16262 : 크레스토정 20mg
16548 : 가바토파정 100mg
16551 : 동아가바펜틴정 800mg
16688 : 오마코연질캡슐(오메가-3-산에틸에스테르90)
18147 : 리리카캡슐 150mg
18357 : 종근당글리아티린연질캡슐(콜린알포세레이트) 
19232 : 콜리네이트연질캡슐 400mg
19552 : 트루비타정 60mg/병
19607 : 스토가정 10mg
19861 : 노바스크정 5mg
20014 : 마도파정
20238 : 플라빅스정 75mg
20877 : 엑스포지정 5/160mg
21325 : 아토르바정 10mg
21771 : 라비에트정 20mg
22074 : 리피로우정 20mg
22347 : 자누비아정 50mg
22362 : 맥시부펜이알정 300mg
24850 : 놀텍정 10mg
25367 : 자누메트정 50/850mg
25438 : 큐시드정 31.5mg/PTP
25469 : 아모잘탄정 5/100mg
27733 : 트윈스타정 40/5mg
27777 : 카나브정 60mg
27926 : 울트라셋이알서방정
27993 : 졸로푸트정 100mg
28763 : 트라젠타정(리나글립틴)
29345 : 비모보정 500/20mg
29451 : 레일라정
29667 : 리바로정 4mg
30308 : 트라젠타듀오정 2.5/850mg
31863 : 아질렉트정(라사길린메실산염)
31885 : 자누메트엑스알서방정 100/1000m

In [11]:
import json
from pathlib import Path

# 기존 56개 클래스 ID
target_ids = {
    1900, 2483, 3351, 3483, 3544, 3743, 3832, 4543,
    12081, 12247, 12778, 13395, 13900, 16232, 16262,
    16548, 16551, 16688, 18147, 18357, 19232, 19552,
    19607, 19861,
    # 아래는 본인이 출력한 나머지 56개 ID도 넣으면 됩니다.
}

aihub_root = Path("../data/aihub/labels")

matched = {}

for json_path in aihub_root.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for img in data.get("images", []):
            drug_n = img.get("drug_N", "")

            if drug_n.startswith("K-"):
                try:
                    drug_id = int(drug_n[2:])

                    if drug_id in target_ids:
                        matched.setdefault(drug_id, 0)
                        matched[drug_id] += 1

                except ValueError:
                    pass

    except Exception:
        continue

print("AI-Hub에서 매칭된 기존 클래스")
print("=" * 60)

for drug_id in sorted(matched):
    print(f"{drug_id}: {matched[drug_id]:,}개 JSON")

print("=" * 60)
print(f"매칭된 클래스 수: {len(matched)} / {len(target_ids)}")

AI-Hub에서 매칭된 기존 클래스
1900: 155개 JSON
2483: 228개 JSON
3544: 261개 JSON
3743: 465개 JSON
3832: 267개 JSON
4543: 336개 JSON
12081: 351개 JSON
12247: 366개 JSON
12778: 447개 JSON
13395: 342개 JSON
13900: 264개 JSON
16232: 807개 JSON
16262: 573개 JSON
16548: 927개 JSON
16551: 917개 JSON
16688: 138개 JSON
18147: 129개 JSON
18357: 135개 JSON
19232: 135개 JSON
19552: 441개 JSON
19607: 201개 JSON
19861: 429개 JSON
매칭된 클래스 수: 22 / 24


In [12]:
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path("..").resolve()

RAW_ANNOTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "sprint_ai_project1_data"
    / "train_annotations"
)

AIHUB_LABEL_DIR = (
    PROJECT_ROOT
    / "data"
    / "aihub"
    / "labels"
)

# ============================================================
# 1. 기존 데이터의 56개 클래스 가져오기
# ============================================================

original_classes = {}

for json_path in RAW_ANNOTATION_DIR.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for category in data.get("categories", []):
            original_classes[category["id"]] = category["name"]

    except Exception:
        continue

print("=" * 70)
print("기존 데이터 클래스 확인")
print("=" * 70)
print(f"기존 클래스 수: {len(original_classes)}")

for category_id, name in sorted(original_classes.items()):
    print(f"{category_id}: {name}")


# ============================================================
# 2. AI-Hub 데이터에서 기존 56개 클래스 찾기
# ============================================================

matched_images = defaultdict(set)
matched_jsons = defaultdict(set)
matched_annotations = defaultdict(int)

parse_errors = 0

print("\n" + "=" * 70)
print("AI-Hub 매칭 시작")
print("=" * 70)

for json_path in AIHUB_LABEL_DIR.rglob("*.json"):

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

    except Exception:
        parse_errors += 1
        continue

    image_info = {}

    # --------------------------------------------------------
    # 이미지 정보 확인
    # --------------------------------------------------------

    for img in data.get("images", []):

        file_name = img.get("file_name")

        if not file_name:
            continue

        drug_n = img.get("drug_N", "")

        if not drug_n.startswith("K-"):
            continue

        try:
            drug_id = int(drug_n[2:])
        except ValueError:
            continue

        # 기존 56개 클래스에 포함되는 경우만
        if drug_id not in original_classes:
            continue

        image_info[img.get("id")] = {
            "drug_id": drug_id,
            "file_name": file_name
        }

        # 같은 이미지가 여러 JSON에 있어도 중복 제거
        matched_images[drug_id].add(file_name)

        matched_jsons[drug_id].add(str(json_path))

    # --------------------------------------------------------
    # Annotation 개수 확인
    # --------------------------------------------------------

    for ann in data.get("annotations", []):

        image_id = ann.get("image_id")

        if image_id not in image_info:
            continue

        drug_id = image_info[image_id]["drug_id"]

        matched_annotations[drug_id] += 1


# ============================================================
# 3. 결과 출력
# ============================================================

print("\n" + "=" * 70)
print("AI-Hub ↔ 기존 56개 클래스 매칭 결과")
print("=" * 70)

total_images = 0
total_annotations = 0
matched_class_count = 0

for category_id in sorted(original_classes):

    image_count = len(matched_images[category_id])
    json_count = len(matched_jsons[category_id])
    annotation_count = matched_annotations[category_id]

    if image_count > 0:
        matched_class_count += 1

    total_images += image_count
    total_annotations += annotation_count

    print(
        f"{category_id:>6} | "
        f"{original_classes[category_id]:<35} | "
        f"이미지 {image_count:>5,}개 | "
        f"JSON {json_count:>5,}개 | "
        f"annotation {annotation_count:>6,}개"
    )


# ============================================================
# 4. 최종 요약
# ============================================================

print("\n" + "=" * 70)
print("최종 요약")
print("=" * 70)

print(f"매칭된 클래스: {matched_class_count} / {len(original_classes)}")
print(f"매칭된 이미지: {total_images:,}개")
print(f"매칭된 annotation: {total_annotations:,}개")
print(f"JSON 파싱 오류: {parse_errors:,}개")

print("=" * 70)

기존 데이터 클래스 확인
기존 클래스 수: 56
1900: 보령부스파정 5mg
2483: 뮤테란캡슐 100mg
3351: 일양하이트린정 2mg
3483: 기넥신에프정(은행엽엑스)(수출용)
3544: 무코스타정(레바미피드)(비매품)
3743: 알드린정
3832: 뉴로메드정(옥시라세탐)
4543: 에어탈정(아세클로페낙)
12081: 리렉스펜정 300mg/PTP
12247: 아빌리파이정 10mg
12778: 다보타민큐정 10mg/병
13395: 써스펜8시간이알서방정 650mg
13900: 에빅사정(메만틴염산염)(비매품)
16232: 리피토정 20mg
16262: 크레스토정 20mg
16548: 가바토파정 100mg
16551: 동아가바펜틴정 800mg
16688: 오마코연질캡슐(오메가-3-산에틸에스테르90)
18147: 리리카캡슐 150mg
18357: 종근당글리아티린연질캡슐(콜린알포세레이트) 
19232: 콜리네이트연질캡슐 400mg
19552: 트루비타정 60mg/병
19607: 스토가정 10mg
19861: 노바스크정 5mg
20014: 마도파정
20238: 플라빅스정 75mg
20877: 엑스포지정 5/160mg
21325: 아토르바정 10mg
21771: 라비에트정 20mg
22074: 리피로우정 20mg
22347: 자누비아정 50mg
22362: 맥시부펜이알정 300mg
24850: 놀텍정 10mg
25367: 자누메트정 50/850mg
25438: 큐시드정 31.5mg/PTP
25469: 아모잘탄정 5/100mg
27733: 트윈스타정 40/5mg
27777: 카나브정 60mg
27926: 울트라셋이알서방정
27993: 졸로푸트정 100mg
28763: 트라젠타정(리나글립틴)
29345: 비모보정 500/20mg
29451: 레일라정
29667: 리바로정 4mg
30308: 트라젠타듀오정 2.5/850mg
31863: 아질렉트정(라사길린메실산염)
31885: 자누메트엑스알서방정 100/1000mg
32310: 글리아타민연질캡슐
33009: 신바로정
3

In [13]:
print("\n" + "=" * 70)
print("클래스별 AI-Hub 매칭 이미지 수")
print("=" * 70)

for category_id in sorted(original_classes):
    image_count = len(matched_images[category_id])

    print(
        f"{category_id:>6} | "
        f"{original_classes[category_id]:<35} | "
        f"{image_count:>6,}개"
    )

print("=" * 70)
print(f"전체 고유 이미지: {sum(len(v) for v in matched_images.values()):,}개")


클래스별 AI-Hub 매칭 이미지 수
  1900 | 보령부스파정 5mg                          |    151개
  2483 | 뮤테란캡슐 100mg                         |    225개
  3351 | 일양하이트린정 2mg                         |      0개
  3483 | 기넥신에프정(은행엽엑스)(수출용)                  |      0개
  3544 | 무코스타정(레바미피드)(비매품)                   |    258개
  3743 | 알드린정                                |    465개
  3832 | 뉴로메드정(옥시라세탐)                        |    267개
  4543 | 에어탈정(아세클로페낙)                        |    335개
 12081 | 리렉스펜정 300mg/PTP                     |    351개
 12247 | 아빌리파이정 10mg                         |    363개
 12778 | 다보타민큐정 10mg/병                       |    447개
 13395 | 써스펜8시간이알서방정 650mg                   |    342개
 13900 | 에빅사정(메만틴염산염)(비매품)                   |    264개
 16232 | 리피토정 20mg                           |    807개
 16262 | 크레스토정 20mg                          |    570개
 16548 | 가바토파정 100mg                         |    921개
 16551 | 동아가바펜틴정 800mg                       |    916개
 16688 | 오마코연질캡슐(오메가-3-산에틸에스테르90)          

In [14]:
print("\n" + "=" * 70)
print("AI-Hub 매칭 이미지 수 (적은 순)")
print("=" * 70)

class_counts = []

for category_id in sorted(original_classes):
    count = len(matched_images[category_id])

    class_counts.append(
        (count, category_id, original_classes[category_id])
    )

for count, category_id, name in sorted(class_counts):
    print(f"{category_id:>6} | {name:<35} | {count:>6,}개")


AI-Hub 매칭 이미지 수 (적은 순)
  3351 | 일양하이트린정 2mg                         |      0개
  3483 | 기넥신에프정(은행엽엑스)(수출용)                  |      0개
 18147 | 리리카캡슐 150mg                         |    129개
 31863 | 아질렉트정(라사길린메실산염)                     |    129개
 32310 | 글리아타민연질캡슐                           |    129개
 18357 | 종근당글리아티린연질캡슐(콜린알포세레이트)              |    135개
 19232 | 콜리네이트연질캡슐 400mg                     |    135개
 20014 | 마도파정                                |    135개
 33880 | 글리틴정(콜린알포세레이트)                      |    135개
 41768 | 카발린캡슐 25mg                          |    135개
 16688 | 오마코연질캡슐(오메가-3-산에틸에스테르90)            |    138개
 22074 | 리피로우정 20mg                          |    144개
 21325 | 아토르바정 10mg                          |    150개
 38162 | 로수바미브정 10/20mg                      |    150개
  1900 | 보령부스파정 5mg                          |    151개
 27993 | 졸로푸트정 100mg                         |    156개
 29345 | 비모보정 500/20mg                       |    191개
 29451 | 레일라정                            

In [15]:
# 클래스별 결과를 파일로 저장
output_path = PROJECT_ROOT / "data" / "aihub_class_counts.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("AI-Hub 매칭 이미지 수 (적은 순)\n")
    f.write("=" * 70 + "\n")

    for count, category_id, name in sorted(class_counts):
        f.write(
            f"{category_id:>6} | "
            f"{name:<40} | "
            f"{count:>6,}개\n"
        )

print(f"저장 완료: {output_path}")

저장 완료: C:\Users\user\git\pill_detection_ai_14\pill-detection-ai\data\aihub_class_counts.txt


In [16]:
from collections import defaultdict

AIHUB_IMAGE_DIR = PROJECT_ROOT / "data" / "aihub" / "images"

# 실제 존재하는 이미지 파일명 목록
existing_images = {
    p.name
    for p in AIHUB_IMAGE_DIR.rglob("*")
    if p.is_file()
}

print("=" * 70)
print("AI-Hub 실제 이미지 파일 확인")
print("=" * 70)

available_images = defaultdict(set)

for category_id in sorted(original_classes):
    for file_name in matched_images[category_id]:

        # 실제 이미지가 존재하는 경우만
        if file_name in existing_images:
            available_images[category_id].add(file_name)

# 결과 출력
total_available = 0

for category_id in sorted(original_classes):
    count = len(available_images[category_id])
    total_available += count

    print(
        f"{category_id:>6} | "
        f"{original_classes[category_id]:<40} | "
        f"{count:>6,}개"
    )

print("=" * 70)
print(f"실제 사용 가능한 이미지: {total_available:,}개")
print("=" * 70)

AI-Hub 실제 이미지 파일 확인
  1900 | 보령부스파정 5mg                               |    151개
  2483 | 뮤테란캡슐 100mg                              |    225개
  3351 | 일양하이트린정 2mg                              |      0개
  3483 | 기넥신에프정(은행엽엑스)(수출용)                       |      0개
  3544 | 무코스타정(레바미피드)(비매품)                        |    252개
  3743 | 알드린정                                     |    465개
  3832 | 뉴로메드정(옥시라세탐)                             |    267개
  4543 | 에어탈정(아세클로페낙)                             |    335개
 12081 | 리렉스펜정 300mg/PTP                          |    351개
 12247 | 아빌리파이정 10mg                              |    357개
 12778 | 다보타민큐정 10mg/병                            |    447개
 13395 | 써스펜8시간이알서방정 650mg                        |    342개
 13900 | 에빅사정(메만틴염산염)(비매품)                        |    264개
 16232 | 리피토정 20mg                                |    807개
 16262 | 크레스토정 20mg                               |    570개
 16548 | 가바토파정 100mg                              |    915개
 16551 | 동아가바펜틴정 800

In [17]:
import random
from collections import defaultdict

MAX_IMAGES_PER_CLASS = 200
SEED = 42

random.seed(SEED)

selected_images = defaultdict(list)

for category_id in sorted(original_classes):

    # 실제 존재하는 이미지 중 선택
    images = list(available_images[category_id])

    # 랜덤 셔플
    random.shuffle(images)

    # 최대 200장
    selected_images[category_id] = images[:MAX_IMAGES_PER_CLASS]


print("=" * 70)
print("AI-Hub 추가 이미지 선택 결과")
print("=" * 70)

total_selected = 0

for category_id in sorted(original_classes):

    count = len(selected_images[category_id])
    total_selected += count

    print(
        f"{category_id:>6} | "
        f"{original_classes[category_id]:<40} | "
        f"{count:>4}개"
    )

print("=" * 70)
print(f"선택된 AI-Hub 이미지: {total_selected:,}개")
print("=" * 70)

AI-Hub 추가 이미지 선택 결과
  1900 | 보령부스파정 5mg                               |  151개
  2483 | 뮤테란캡슐 100mg                              |  200개
  3351 | 일양하이트린정 2mg                              |    0개
  3483 | 기넥신에프정(은행엽엑스)(수출용)                       |    0개
  3544 | 무코스타정(레바미피드)(비매품)                        |  200개
  3743 | 알드린정                                     |  200개
  3832 | 뉴로메드정(옥시라세탐)                             |  200개
  4543 | 에어탈정(아세클로페낙)                             |  200개
 12081 | 리렉스펜정 300mg/PTP                          |  200개
 12247 | 아빌리파이정 10mg                              |  200개
 12778 | 다보타민큐정 10mg/병                            |  200개
 13395 | 써스펜8시간이알서방정 650mg                        |  200개
 13900 | 에빅사정(메만틴염산염)(비매품)                        |  200개
 16232 | 리피토정 20mg                                |  200개
 16262 | 크레스토정 20mg                               |  200개
 16548 | 가바토파정 100mg                              |  200개
 16551 | 동아가바펜틴정 800mg                            | 